# Task 1: Word occurrence counting with Pthreads

6CS005 High Performance Computing. The assessed implementation is the separate C/CUDA source in the public repository. Python cells only download, compile, execute, validate, and display evidence.

Run the cells in order. Inputs are public; no personal Google Drive is mounted. Record actual execution on your PC after all checks pass.


## 1. Fetch the exact source and input revision


In [ ]:
import os, sys, json, subprocess, hashlib, platform
from pathlib import Path
REVISION = "0bc5ad74e09d3edf931beff420a6170fdaf18337"
ROOT = Path("/content/hpc_" + REVISION[:12])
if not ROOT.exists():
    subprocess.run(["git","clone","--no-checkout","https://github.com/Ihsan619/hpc.git",str(ROOT)],check=True)
    subprocess.run(["git","checkout",REVISION],cwd=ROOT,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=ROOT,text=True).strip() == REVISION
os.chdir(ROOT)
manifest = json.loads((ROOT/"docs/input_manifest.json").read_text())
for name,record in manifest.items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest() == record["sha256"], name
print("Public source revision:", REVISION)
print("Input SHA-256 checks passed:", len(manifest))
environment = {"revision":REVISION,"platform":platform.platform(),"python":sys.version}
for command in [["gcc","--version"],["lscpu"],["nvcc","--version"],["nvidia-smi"]]:
    try:
        result=subprocess.run(command,text=True,capture_output=True)
        environment[" ".join(command)] = result.stdout + result.stderr
        print(environment[" ".join(command)])
    except FileNotFoundError:
        environment[" ".join(command)] = "unavailable"
(ROOT/"evidence").mkdir(exist_ok=True)
(ROOT/"evidence/task1_environment.json").write_text(json.dumps(environment,indent=2))


## 2. Compile the standalone source


In [ ]:
subprocess.run(['make','task1/word_count'],cwd=ROOT,check=True)


## 3. Run the task and inspect its output


In [ ]:
os.chdir(ROOT/"task1")
command = ['./word_count','data/WordOccurrenceDataset.txt','4']
print("Command:", " ".join(command))
result = subprocess.run(command,text=True,capture_output=True,check=True)
print(result.stdout)
(ROOT/"evidence/task1_demonstration.txt").write_text(result.stdout)


In [ ]:
output = Path("result.txt").read_text().splitlines()
print("Output file: result.txt | Lines:",len(output))
print("\n".join(output[:45]))


## 4. Independent correctness checks and repeated timing trials

Validation checks normal and invalid inputs. Three measured trials follow one excluded warm-up per configuration. Process wall time includes I/O and startup; the C/CUDA console reports the narrower compute/kernel timing. Do not describe these as interchangeable measurements.


In [ ]:
os.chdir(ROOT)
subprocess.run([sys.executable,"validation/validate.py","task1","--benchmark"],check=True)
record = json.loads((ROOT/"evidence/task1_validation.json").read_text())
assert record["passed"]
print("Independent verification: PASS")
import statistics
groups = {}
for trial in record["benchmark_trials"]:
    groups.setdefault(trial["threads"],[]).append(trial["process_wall_seconds"])
for threads,values in groups.items():
    print("Threads:",threads,"| process wall median:",round(statistics.median(values),6),"s | trials:",values)


## 5. Download actual outputs and evidence

This downloads generated files, validation records, and runtime details. Keep screen recordings separately under the local project evidence folder.


In [ ]:
import zipfile
bundle=Path('/content/task1_outputs_and_evidence.zip')
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in (ROOT/'evidence').glob('task1_*'):
        archive.write(path,path.relative_to(ROOT))
    archive.write(ROOT/"task1/result.txt","task1/result.txt")
print("Bundle:",bundle)
from google.colab import files
files.download(str(bundle))
